In [2]:
!pip install -qU transformers sentence-transformers faiss-cpu

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

In [7]:
# 1. raw unstructured data
sentences = [
    "The Eiffel Tower stands tall in the heart of Paris, France.",
    "I'm Priyabrata from West Bengal.",
    "Coffee is one of the most widely consumed beverages in the world.",
    "The sun rises in the east and sets in the west every single day.",
    "I works at KPIT as SSE.",
    "Subhendu Adhikari has been elected as the Chief Minister of West Bengal.",
    "A balanced diet and regular exercise are key to maintaining good health.",
    "I didn't get any increment this year either at KPIT.",
    "Blockchain technology is revolutionizing the way financial transactions are recorded.",
    "I have completed my masters from IIT Mandi in 2024.",
    "The Indian Institute of Technology Mandi is a premier research university located in Kamand, Himachal Pradesh."
]

In [5]:
# 2. Load the Embedding Model (The Librarian)
print("Loading BGE-M3 model...")
model = SentenceTransformer('BAAI/bge-m3')
print("Model loaded successfully.")

Loading BGE-M3 model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded successfully.


In [8]:
# 3. Generate Embeddings
print("Encoding sentences...")
embeddings = model.encode(sentences)

# Inspect the architectural output
print(f"Total sentences: {len(sentences)}")
print(f"Embedding Array Shape: {embeddings.shape}")
print(f"Data Type: {embeddings.dtype}")

# Look at a snippet of the first vector
print(f"First 5 dimensions of sentence 0: {embeddings[0][:5]}")

Encoding sentences...
Total sentences: 11
Embedding Array Shape: (11, 1024)
Data Type: float32
First 5 dimensions of sentence 0: [ 0.01734946  0.01370495 -0.01674865  0.02454143  0.00609526]


In [10]:
# 4. Create the vector DB
dimension_size = embeddings.shape[1]

vector_database = faiss.IndexFlatIP(dimension_size)

vector_database.add(embeddings)

print(f"Total vectors stored in DB : {vector_database.ntotal}")

Total vectors stored in DB : 11


In [12]:
# 5. Execute semantic search
query = "Priyabrata"

query_embedding =model.encode([query])
k = 11 # I want to see top 5 results for respected query
distances, indices = vector_database.search(query_embedding,k)
print("Query Searching ......\n")
for i in range(k):
  match_index = indices[0][i]
  score = distances[0][i]
  print(f"Rank -> {i+1} (Score : {score}) : {sentences[match_index]}")


Query Searching ......

Rank -> 1 (Score : 0.7114197015762329) : I'm Priyabrata from West Bengal.
Rank -> 2 (Score : 0.29558753967285156) : I works at KPIT as SSE.
Rank -> 3 (Score : 0.25834035873413086) : I have completed my masters from IIT Mandi in 2024.
Rank -> 4 (Score : 0.25688257813453674) : The sun rises in the east and sets in the west every single day.
Rank -> 5 (Score : 0.2472321093082428) : A balanced diet and regular exercise are key to maintaining good health.
Rank -> 6 (Score : 0.24648116528987885) : Subhendu Adhikari has been elected as the Chief Minister of West Bengal.
Rank -> 7 (Score : 0.24213039875030518) : I didn't get any increment this year either at KPIT.
Rank -> 8 (Score : 0.23730874061584473) : Blockchain technology is revolutionizing the way financial transactions are recorded.
Rank -> 9 (Score : 0.21462860703468323) : The Eiffel Tower stands tall in the heart of Paris, France.
Rank -> 10 (Score : 0.20369091629981995) : Coffee is one of the most widely consu